# 🤖 Lab 2: Machine Learning - Classificação

**Objetivo**: Construir e avaliar modelos de Machine Learning para classificação.

---

## 🎯 O que você vai aprender:
1. Preparar dados para ML
2. Treinar modelos de classificação
3. Avaliar performance dos modelos
4. Comparar diferentes algoritmos
5. Fazer previsões

**Caso de uso real**: Prever se um cliente vai cancelar o serviço (Churn Prediction)

**Tempo estimado**: 25 minutos

---

## 1️⃣ Importando Bibliotecas

In [ ]:
# Bibliotecas básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Bibliotecas de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# Algoritmos de ML
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Configurações
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')

print("✅ Bibliotecas importadas com sucesso!")

## 2️⃣ Criando Dataset de Clientes

Vamos criar um dataset simulado de uma empresa de telecomunicações.

In [ ]:
np.random.seed(42)
n_clientes = 1000

# Criando features (características) dos clientes
dados = {
    'tempo_cliente_meses': np.random.randint(1, 72, n_clientes),
    'idade': np.random.randint(18, 70, n_clientes),
    'plano': np.random.choice(['Básico', 'Premium', 'Elite'], n_clientes),
    'valor_mensal': np.random.uniform(50, 200, n_clientes),
    'chamadas_suporte': np.random.poisson(2, n_clientes),
    'satisfacao': np.random.randint(1, 6, n_clientes),
    'uso_dados_gb': np.random.gamma(2, 5, n_clientes)
}

df = pd.DataFrame(dados)

# Criando target: Cliente vai cancelar?
# Regra de negócio simulada
df['cancelou'] = (
    (df['tempo_cliente_meses'] < 12) & 
    (df['chamadas_suporte'] > 3) |
    (df['satisfacao'] <= 2) |
    ((df['valor_mensal'] > 150) & (df['uso_dados_gb'] < 5))
).astype(int)

print("✅ Dataset criado!")
print(f"📊 Total de clientes: {len(df):,}")
print(f"❌ Taxa de churn: {df['cancelou'].mean()*100:.1f}%")
print(f"✓ Clientes ativos: {(1-df['cancelou'].mean())*100:.1f}%")

In [ ]:
# Visualizando os dados
display(df.head(10))
df.info()

## 3️⃣ Análise Exploratória Rápida

In [ ]:
# Distribuição da variável alvo
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Gráfico 1: Distribuição de Churn
df['cancelou'].value_counts().plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Distribuição de Churn')
axes[0].set_xlabel('Cancelou (0=Não, 1=Sim)')
axes[0].set_ylabel('Quantidade')
axes[0].set_xticklabels(['Ativo', 'Cancelou'], rotation=0)

# Gráfico 2: Satisfação vs Churn
df.groupby(['satisfacao', 'cancelou']).size().unstack().plot(kind='bar', ax=axes[1], stacked=True)
axes[1].set_title('Satisfação vs Churn')
axes[1].set_xlabel('Nível de Satisfação')
axes[1].legend(['Ativo', 'Cancelou'])

# Gráfico 3: Valor Mensal vs Churn
df.boxplot(column='valor_mensal', by='cancelou', ax=axes[2])
axes[2].set_title('Valor Mensal vs Churn')
axes[2].set_xlabel('Cancelou')
axes[2].set_ylabel('Valor Mensal (R$)')

plt.tight_layout()
plt.show()

## 4️⃣ Preparação dos Dados

**Etapa crucial**: Machine Learning precisa de dados numéricos e normalizados!

In [ ]:
# Separando features (X) e target (y)
X = df.drop('cancelou', axis=1)
y = df['cancelou']

# Convertendo variável categórica 'plano' em números
le = LabelEncoder()
X['plano'] = le.fit_transform(X['plano'])

print("📋 Features (X):")
print(X.columns.tolist())
print(f"\n📊 Shape de X: {X.shape}")
print(f"📊 Shape de y: {y.shape}")

### Dividindo em Treino e Teste

**Regra de ouro**: NUNCA treinar e testar com os mesmos dados!

In [ ]:
# Dividindo: 80% treino, 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"🎯 Treino: {len(X_train)} clientes ({len(X_train)/len(X)*100:.0f}%)")
print(f"🎯 Teste:  {len(X_test)} clientes ({len(X_test)/len(X)*100:.0f}%)")
print(f"\n📊 Distribuição de churn no treino: {y_train.mean()*100:.1f}%")
print(f"📊 Distribuição de churn no teste:  {y_test.mean()*100:.1f}%")

### Normalização dos Dados

Colocando todas as features na mesma escala para melhor performance.

In [ ]:
# Normalizando (StandardScaler: média=0, desvio=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Dados normalizados!")
print(f"Média após normalização: {X_train_scaled.mean():.4f}")
print(f"Desvio padrão após normalização: {X_train_scaled.std():.4f}")

## 5️⃣ Treinando Modelos de ML

Vamos treinar 4 algoritmos diferentes e compará-los!

In [ ]:
# Dicionário de modelos
modelos = {
    'Regressão Logística': LogisticRegression(random_state=42),
    'Árvore de Decisão': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(random_state=42, probability=True)
}

# Treinando todos os modelos
resultados = {}

print("🔄 Treinando modelos...\n")
for nome, modelo in modelos.items():
    # Treinar
    modelo.fit(X_train_scaled, y_train)
    
    # Prever
    y_pred = modelo.predict(X_test_scaled)
    
    # Calcular métricas
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    resultados[nome] = {
        'modelo': modelo,
        'predicoes': y_pred,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1
    }
    
    print(f"✅ {nome}")
    print(f"   Accuracy:  {acc:.3f}")
    print(f"   Precision: {prec:.3f}")
    print(f"   Recall:    {rec:.3f}")
    print(f"   F1-Score:  {f1:.3f}\n")

## 6️⃣ Comparando Modelos Visualmente

In [ ]:
# Criando DataFrame com resultados
df_resultados = pd.DataFrame({
    'Modelo': list(resultados.keys()),
    'Accuracy': [r['accuracy'] for r in resultados.values()],
    'Precision': [r['precision'] for r in resultados.values()],
    'Recall': [r['recall'] for r in resultados.values()],
    'F1-Score': [r['f1_score'] for r in resultados.values()]
})

# Visualizando comparação
df_resultados_plot = df_resultados.set_index('Modelo')
df_resultados_plot.plot(kind='bar', figsize=(12, 6), rot=45)
plt.title('📊 Comparação de Performance dos Modelos', fontsize=14, fontweight='bold')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Tabela Resumo:")
display(df_resultados.round(3))

## 7️⃣ Matriz de Confusão

**Entendendo os erros do modelo**

In [ ]:
# Pegando o melhor modelo (Random Forest)
melhor_modelo = 'Random Forest'
y_pred_melhor = resultados[melhor_modelo]['predicoes']

# Calculando matriz de confusão
cm = confusion_matrix(y_test, y_pred_melhor)

# Visualizando
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Ativo', 'Cancelou'],
            yticklabels=['Ativo', 'Cancelou'])
plt.title(f'Matriz de Confusão - {melhor_modelo}', fontsize=14, fontweight='bold')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predito')
plt.show()

print("\n📊 Interpretação da Matriz:")
print(f"✅ Verdadeiros Negativos (correto 'ativo'): {cm[0,0]}")
print(f"❌ Falsos Positivos (erro: previu cancelamento): {cm[0,1]}")
print(f"❌ Falsos Negativos (erro: não previu cancelamento): {cm[1,0]}")
print(f"✅ Verdadeiros Positivos (correto 'cancelou'): {cm[1,1]}")

## 8️⃣ Fazendo Previsões com Novos Clientes

In [ ]:
# Simulando 3 novos clientes
novos_clientes = pd.DataFrame({
    'tempo_cliente_meses': [6, 36, 48],
    'idade': [25, 45, 52],
    'plano': ['Básico', 'Premium', 'Elite'],
    'valor_mensal': [180, 120, 95],
    'chamadas_suporte': [5, 1, 0],
    'satisfacao': [2, 4, 5],
    'uso_dados_gb': [3, 15, 25]
})

print("🆕 Novos Clientes para Análise:\n")
display(novos_clientes)

# Preparando dados (mesma transformação do treino)
novos_clientes['plano'] = le.transform(novos_clientes['plano'])
novos_clientes_scaled = scaler.transform(novos_clientes)

# Fazendo previsões
modelo_final = resultados[melhor_modelo]['modelo']
predicoes = modelo_final.predict(novos_clientes_scaled)
probabilidades = modelo_final.predict_proba(novos_clientes_scaled)

# Mostrando resultados
print("\n🎯 Previsões:\n")
for i in range(len(novos_clientes)):
    status = "⚠️ VAI CANCELAR" if predicoes[i] == 1 else "✅ CLIENTE ATIVO"
    prob_cancelar = probabilidades[i][1] * 100
    print(f"Cliente {i+1}: {status} (Probabilidade de churn: {prob_cancelar:.1f}%)")

## 🎯 Desafio Prático!

Agora é sua vez de experimentar:

1. **Mude os hiperparâmetros** dos modelos (ex: `max_depth` da árvore)
2. **Crie novas features** combinando as existentes
3. **Teste outros algoritmos** (XGBoost, KNN, etc.)
4. **Analise importância das features** no Random Forest

In [ ]:
# 💻 SEU CÓDIGO AQUI!
# Exemplo: Importância das features
modelo_rf = resultados['Random Forest']['modelo']
importancias = pd.DataFrame({
    'Feature': X.columns,
    'Importância': modelo_rf.feature_importances_
}).sort_values('Importância', ascending=False)

print("📊 Importância das Features:\n")
display(importancias)

# Visualizando
plt.figure(figsize=(10, 6))
plt.barh(importancias['Feature'], importancias['Importância'])
plt.xlabel('Importância')
plt.title('Importância das Features no Random Forest')
plt.tight_layout()
plt.show()

## ✅ Conclusão do Lab 2

**Parabéns!** 🎉 Você construiu e avaliou modelos de Machine Learning!

### O que você aprendeu:
- ✅ Preparar dados para ML (encoding, scaling, split)
- ✅ Treinar múltiplos algoritmos
- ✅ Avaliar modelos com métricas
- ✅ Comparar performance
- ✅ Fazer previsões com novos dados

### Próximo passo:
➡️ **Lab 3**: Deep Learning com Redes Neurais!

---

**💡 Dica profissional**: *"O modelo perfeito não existe. O importante é escolher o melhor para SEU problema específico!"*
